In [ ]:
# ==============================================================================
# 📚 데이터셋 분석 및 실습 목표
# ==============================================================================
# 데이터셋명: Bingsu/zeroth-korean
# 한글 제목: 제로스-코리안 (Zeroth-Korean)
# 의미/설명: 한국어 음성 인식(ASR)을 위한 대규모 코퍼스 데이터셋입니다.
#          'audio'와 'text'가 쌍을 이루어 저장되어 있으며, 사용자는 주어진 음성 파일을 텍스트로 변환하는 연습을 할 수 있습니다.
#          (즉, 음성 파일과 그에 맞는 스크립트가 쌍으로 제공됩니다.)
#
# ✨ 실습 목표: ASR 데이터를 다루는 초보자가 데이터의 기본 구조를 파악하고,
#               텍스트와 오디오 데이터가 어떻게 결합되어 있는지 분석하는 과정을 경험합니다.
# ------------------------------------------------------------------------------

import numpy as np
import random
from datasets import load_dataset, Dataset
import time

# 데이터셋 ID 정의
DATASET_ID = "Bingsu/zeroth-korean"
# 로드할 스플릿 (훈련 데이터만 샘플링하여 사용)
TARGET_SPLIT = 'train'
# 실습에 사용할 샘플 개수
SAMPLE_COUNT = 10

print("=================================================================")
print(f"🔑 [Step 1] 데이터셋 로드 및 스트리밍 가능 여부 확인 (데이터셋: {DATASET_ID})")
print("=================================================================\n")

# ------------------------------------------------------------------------------
# 💡 [필수 구현] 데이터셋 로드 로직 (스트리밍 우선, 실패 시 일반 로드)
# ------------------------------------------------------------------------------
dataset = None
try:
    # 1. 스트리밍 모드로 시도 (가장 빠름)
    print("🚀 스트리밍 모드(streaming=True)로 데이터 로드를 시도합니다...")
    dataset = load_dataset(DATASET_ID, split=TARGET_SPLIT, streaming=True)
    print("✅ 스트리밍 모드 로드 성공! (IterableDataset)")
except Exception as e:
    # 스트리밍 로드가 실패하면 일반 로드로 전환
    print(f"⚠️ 스트리밍 로드 실패 ({type(e).__name__}). 일반 데이터셋(streaming=False)으로 전환합니다.")
    try:
        dataset = load_dataset(DATASET_ID, split=TARGET_SPLIT)
        print("✅ 일반 데이터셋 로드 성공! (Dataset)")
    except Exception as e_fallback:
        print(f"❌ 모든 로드 시도 실패. 데이터셋 로드에 문제가 있습니다. 에러: {e_fallback}")
        exit()

print("\n" + "="*80)
print(f"🎉 [Step 2] {SAMPLE_COUNT}개의 샘플 데이터를 확보했습니다. (전체 데이터셋 구조 파악)")
print("="*80)

# ------------------------------------------------------------------------------
# 📚 데이터셋 샘플링 로직 (스트리밍 vs 일반 데이터셋 처리 분기)
# ------------------------------------------------------------------------------

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    print("✅ [인식] 스트리밍 데이터셋 (IterableDataset)으로 처리합니다.")
    # 스트리밍 데이터셋은 next()와 take()를 사용해 개별적으로 접근해야 합니다.
    sample_dataset_iterator = dataset.take(SAMPLE_COUNT)
    
    # 리스트로 변환하여 반복문 처리를 용이하게 합니다.
    sample_data_list = []
    for _ in range(min(SAMPLE_COUNT, 50)): # 혹시 모를 무한 루프 방지
        try:
            sample_data_list.append(next(sample_dataset_iterator))
        except StopIteration:
            break
else:
    # 일반 데이터셋 (Dataset)
    print("✅ [인식] 일반 데이터셋 (Dataset)으로 처리합니다.")
    sample_data_list = list(dataset.take(SAMPLE_COUNT))

# ------------------------------------------------------------------------------
# 🛠️ [실습 1] 데이터 구조 분석: 오디오와 텍스트의 관계 확인
# ------------------------------------------------------------------------------
print("\n\n" + "="*80)
print("✨ [실습 1] 데이터 구조 분석: 오디오 파일의 속성 확인")
print("="*80)

print(f"💡 주석: 이 데이터셋의 핵심은 '음성 파일'과 '스크립트(text)'가 쌍을 이루는 것입니다.")
print("    우리는 이 샘플들을 통해 데이터의 형식을 이해합니다.")

audio_feature_name = 'audio'
text_feature_name = 'text'

print(f"\n-> 샘플 데이터 {len(sample_data_list)}개에 대한 구조 분석:")

for i, sample in enumerate(sample_data_list):
    # 오디오 데이터 접근 및 확인
    if audio_feature_name in sample:
        audio_info = sample[audio_feature_name]
        print(f"  [{i+1}] 🔊 오디오 정보: (Sampling Rate: {audio_info['sampling_rate']} Hz, Type: {type(audio_info['array']).__name__})")
    
    # 텍스트 데이터 접근 및 확인
    if text_feature_name in sample:
        text_content = sample[text_feature_name]
        # 텍스트의 길이(글자 수)를 계산하여 분석 요소로 사용
        text_length = len(str(text_content))
        print(f"  [{i+1}] 📝 텍스트 내용: '{text_content[:30]}...' (글자 수: {text_length}자)")
    
# ------------------------------------------------------------------------------
# 📊 [실습 2] 창의적 탐색: 텍스트 길이와 평균 길이 분포 계산
# ------------------------------------------------------------------------------
print("\n\n" + "="*80)
print("📊 [실습 2] 데이터 탐색: 샘플 텍스트 길이의 통계적 분포 분석")
print("=================================================================")
print("💡 주석: 데이터셋의 전반적인 텍스트 길이가 얼마나 되는지, 분포를 확인합니다.")

text_lengths = []
for sample in sample_data_list:
    if text_feature_name in sample:
        text_content = sample[text_feature_name]
        text_lengths.append(len(str(text_content)))

if text_lengths:
    # 기본적인 정량적 분석 수행
    avg_length = np.mean(text_lengths)
    max_length = np.max(text_lengths)
    min_length = np.min(text_lengths)
    
    print(f"✅ 분석된 샘플 {len(text_lengths)}개의 텍스트 길이를 기반으로:")
    print(f"    -> 최소 길이 (Min): {min_length} 글자")
    print(f"    -> 최대 길이 (Max): {max_length} 글자")
    print(f"    -> 평균 길이 (Mean): {avg_length:.2f} 글자")
    print("\n(해석: 이 데이터셋의 일반적인 문장 길이는 평균적으로 약 {:.2f} 글자 수준인 것으로 추정됩니다.)".format(avg_length))
else:
    print("❌ 분석할 텍스트 데이터가 충분하지 않습니다.")

# ------------------------------------------------------------------------------
# 🧑‍💻 [실습 3] 간단한 시뮬레이션: 가장 긴 텍스트 3개 추출하기
# ------------------------------------------------------------------------------
print("\n\n" + "="*80)
print("🚀 [실습 3] 실용 시뮬레이션: 가장 긴 문장 3개 추출하여 비교")
print("=================================================================")
print("💡 주석: 실제 ASR 시스템에서는 '어떤 문장이 더 긴가?'가 중요할 수 있습니다.")

# 텍스트 길이를 저장하고, 정렬할 리스트 생성
sorted_samples = []
for sample in sample_data_list:
    if text_feature_name in sample:
        text_content = sample[text_feature_name]
        length = len(str(text_content))
        # (길이, 텍스트 내용) 튜플로 저장하여 길이 기준으로 정렬 가능하게 함
        sorted_samples.append((length, text_content))

# 길이가 긴 순서대로 정렬 (내림차순)
sorted_samples.sort(key=lambda x: x[0], reverse=True)

print(f"🏆 가장 긴 텍스트 샘플 {min(3, len(sorted_samples))}개를 찾았습니다:")

for i, (length, content) in enumerate(sorted_samples[:3]):
    print(f"\n[{i+1}] (길이: {length}자)")
    print(f"    - 📜 텍스트: {content}")
    
print("\n=================================================================")
print("✨ 실습 완료! 이 데이터셋은 음성 데이터를 다루므로, 실제 활용을 위해서는")
print("   음성 파일(audio)을 불러와 처리하는 전문 라이브러리(Librosa 등) 지식이 추가로 필요합니다.")
print("=================================================================")